In [1]:
import pandas as pd
import geopandas as gpd
import folium
import numpy as np

# 1. INGESTA DE DATOS ESPACIALES (IDS)
# IMPORTANTE: Cambia 'ids_cdmx.shp' por el nombre exacto de tu archivo shapefile
gdf_ids = gpd.read_file('../data/raw/ids_cdmx.shp')

# 2. INGESTA Y LIMPIEZA DEL CENSO (INEGI)
# CORRECCIÓN: Leemos directamente desde la fila 1 sin saltar metadatos
df_censo = pd.read_excel('../data/raw/cpv2020.xlsx')

# Limpieza de confidencialidad del INEGI (los asteriscos)
df_censo.replace({'*': np.nan, 'N/D': np.nan}, inplace=True)

# Seleccionamos variables (ahora sí las va a encontrar)
columnas_interes = ['ENTIDAD', 'MUN', 'LOC', 'AGEB', 'POBTOT', 'GRAPROES']
df_censo = df_censo[columnas_interes]

# Reconstrucción del CVEGEO (13 dígitos) para asegurar el cruce exacto
df_censo['CVEGEO'] = (
    df_censo['ENTIDAD'].astype(str).str.zfill(2) + 
    df_censo['MUN'].astype(str).str.zfill(3) + 
    df_censo['LOC'].astype(str).str.zfill(4) + 
    df_censo['AGEB'].astype(str).str.zfill(4)
)

df_censo = df_censo.drop_duplicates(subset=['CVEGEO'], keep='first')
# Convertimos la escolaridad a valores numéricos
df_censo['GRAPROES'] = pd.to_numeric(df_censo['GRAPROES'], errors='coerce')

# 3. FUSIÓN ESPACIAL (MERGE)
# Estandarizamos el nombre de la llave en el Censo para que sea idéntico al Shapefile
df_censo.rename(columns={'CVEGEO': 'cvegeo'}, inplace=True)

# Ahora el cruce es perfectamente simétrico
gdf_master = gdf_ids.merge(df_censo[['cvegeo', 'POBTOT', 'GRAPROES']], on='cvegeo', how='inner')

# 4. REPROYECCIÓN Y OPTIMIZACIÓN ESPACIAL (CRÍTICO)
gdf_master = gdf_master.to_crs(epsg=4326)

# Algoritmo de simplificación de Douglas-Peucker. 
# 0.0005 grados son aprox 50 metros. Reduce el HTML de 225MB a ~10MB.
gdf_master['geometry'] = gdf_master.simplify(0.0005)

# 5. RENDERIZADO DEL MAPA CLASE MUNDIAL (FOLIUM)
mapa_cdmx = folium.Map(location=[19.4326, -99.1332], zoom_start=11, tiles='CartoDB dark_matter')

folium.Choropleth(
    geo_data=gdf_master,
    name='Grado Promedio de Escolaridad',
    data=gdf_master,
    columns=['cvegeo', 'GRAPROES'],
    key_on='feature.properties.cvegeo',
    fill_color='YlGnBu',
    fill_opacity=0.7,
    line_opacity=0.2,
    nan_fill_color='black',
    legend_name='Años de Escolaridad Promedio'
).add_to(mapa_cdmx)

mapa_cdmx.save('../docs/index.html')
print("Mapa renderizado, optimizado y exportado con éxito.")

Mapa renderizado, optimizado y exportado con éxito.


In [2]:
import libpysal as lps
from esda.moran import Moran_Local
import numpy as np

# 1. LIMPIEZA ESTRICTA
# La econometría espacial no tolera valores nulos. Filtramos las AGEBs sin datos.
gdf_lisa = gdf_master.dropna(subset=['GRAPROES']).copy()

# 2. MATRIZ DE PESOS ESPACIALES (Fricción Topológica)
# Usamos contigüidad tipo "Reina" (Queen): si dos AGEBs comparten un vértice o lado, son vecinas.
wq = lps.weights.Queen.from_dataframe(gdf_lisa)
wq.transform = 'r' # Estandarización por filas para promediar el efecto de los vecinos

# 3. CÁLCULO DEL I DE MORAN LOCAL (LISA)
y = gdf_lisa['GRAPROES'].values
moran_loc = Moran_Local(y, wq)

# 4. CLASIFICACIÓN DEL SISTEMA CENTRO-PERIFERIA (Nivel de significancia: 5%)
# Asignamos el cuadrante a cada AGEB basado en su significancia estadística (p-value < 0.05)
sig = 0.05
spots = ['No Significativo', 'High-High (Centro Integrado)', 'Low-Low (Trampa Periférica)', 
         'Low-High (Transición)', 'High-Low (Oasis)']
labels = [spots[i] for i in moran_loc.q]

gdf_lisa['cluster_lisa'] = labels
gdf_lisa.loc[moran_loc.p_sim > sig, 'cluster_lisa'] = 'No Significativo'

# 5. AUDITORÍA DE RESULTADOS
print("--- DISTRIBUCIÓN ESTRUCTURAL DE LA DEPENDENCIA (AGEBs) ---")
print(gdf_lisa['cluster_lisa'].value_counts())

C:\Users\Jose\AppData\Local\Temp\ipykernel_15992\3211665174.py:11: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  wq = lps.weights.Queen.from_dataframe(gdf_lisa)
c:\Users\Jose\ids-vs-educacion-cdmx\venv\Lib\site-packages\libpysal\weights\contiguity.py:354: UserWarning: The weights matrix is not fully connected: 
 There are 30 disconnected components.
 There are 18 islands with ids: 242, 301, 314, 325, 372, 380, 398, 685, 952, 970, 1367, 1408, 1442, 1628, 1633, 1639, 1640, 2168.
  W.__init__(self, neighbors, ids=ids, **kw)


('WARNING: ', 242, ' is an island (no neighbors)')
('WARNING: ', 301, ' is an island (no neighbors)')
('WARNING: ', 314, ' is an island (no neighbors)')
('WARNING: ', 325, ' is an island (no neighbors)')
('WARNING: ', 372, ' is an island (no neighbors)')
('WARNING: ', 380, ' is an island (no neighbors)')
('WARNING: ', 398, ' is an island (no neighbors)')
('WARNING: ', 685, ' is an island (no neighbors)')
('WARNING: ', 952, ' is an island (no neighbors)')
('WARNING: ', 970, ' is an island (no neighbors)')
('WARNING: ', 1367, ' is an island (no neighbors)')
('WARNING: ', 1408, ' is an island (no neighbors)')
('WARNING: ', 1442, ' is an island (no neighbors)')
('WARNING: ', 1628, ' is an island (no neighbors)')
('WARNING: ', 1633, ' is an island (no neighbors)')
('WARNING: ', 1639, ' is an island (no neighbors)')
('WARNING: ', 1640, ' is an island (no neighbors)')
('WARNING: ', 2168, ' is an island (no neighbors)')
--- DISTRIBUCIÓN ESTRUCTURAL DE LA DEPENDENCIA (AGEBs) ---
cluster_lisa
No

c:\Users\Jose\ids-vs-educacion-cdmx\venv\Lib\site-packages\esda\moran.py:1398: RuntimeWarning: invalid value encountered in divide
  self.z_sim = (self.Is - self.EI_sim) / self.seI_sim


In [4]:
import libpysal as lps
from esda.moran import Moran_Local
import numpy as np
import folium

# 1. MATRIZ TOPOLÓGICA Y SEMILLA DETERMINISTA
gdf_lisa = gdf_master.dropna(subset=['GRAPROES']).copy()
wq = lps.weights.Queen.from_dataframe(gdf_lisa)
wq.transform = 'r'
y = gdf_lisa['GRAPROES'].values

# Fijamos la semilla para anular la variación estocástica (Rigor matemático)
np.random.seed(12345) 
moran_loc = Moran_Local(y, wq)

# 2. CLASIFICACIÓN DE LA MICROESTRUCTURA
sig = 0.05
spots = ['No Significativo (Normalidad Urbana)', 'High-High (Centro Integrado)', 
         'Low-Low (Trampa Periférica)', 'Low-High (Transición)', 'High-Low (Oasis)']
labels = [spots[i] for i in moran_loc.q]

gdf_lisa['cluster_lisa'] = labels
gdf_lisa.loc[moran_loc.p_sim > sig, 'cluster_lisa'] = 'No Significativo (Normalidad Urbana)'

# 3. PALETA ESPACIAL
color_dict = {
    'High-High (Centro Integrado)': '#1f77b4',  # Azul oscuro
    'Low-Low (Trampa Periférica)': '#d62728',   # Rojo
    'Low-High (Transición)': '#aec7e8',         # Azul claro
    'High-Low (Oasis)': '#ffbb78',              # Naranja/Rosa
    'No Significativo (Normalidad Urbana)': '#555555' # Gris visible
}
gdf_lisa['color'] = gdf_lisa['cluster_lisa'].map(color_dict)

# 4. RENDERIZADO DEL MAPA WEB
mapa_lisa = folium.Map(location=[19.4326, -99.1332], zoom_start=11, tiles='CartoDB dark_matter')

folium.GeoJson(
    gdf_lisa,
    name='Clústeres LISA de Dependencia Educativa',
    style_function=lambda feature: {
        'fillColor': feature['properties']['color'],
        'color': '#000000',
        'weight': 0.3,
        # Damos un 80% de opacidad a los extremos y 30% a la normalidad para generar contraste visual
        'fillOpacity': 0.8 if 'Normalidad' not in feature['properties']['cluster_lisa'] else 0.3
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['cvegeo', 'GRAPROES', 'cluster_lisa'],
        aliases=['CVEGEO AGEB:', 'Años Escolaridad:', 'Clúster Espacial:'],
        localize=True
    )
).add_to(mapa_lisa)

mapa_lisa.save('../docs/lisa_map.html')
print("Mapa econométrico LISA renderizado con éxito en 'docs/lisa_map.html'.")

C:\Users\Jose\AppData\Local\Temp\ipykernel_15992\1940765769.py:8: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  wq = lps.weights.Queen.from_dataframe(gdf_lisa)
c:\Users\Jose\ids-vs-educacion-cdmx\venv\Lib\site-packages\libpysal\weights\contiguity.py:354: UserWarning: The weights matrix is not fully connected: 
 There are 30 disconnected components.
 There are 18 islands with ids: 242, 301, 314, 325, 372, 380, 398, 685, 952, 970, 1367, 1408, 1442, 1628, 1633, 1639, 1640, 2168.
  W.__init__(self, neighbors, ids=ids, **kw)


('WARNING: ', 242, ' is an island (no neighbors)')
('WARNING: ', 301, ' is an island (no neighbors)')
('WARNING: ', 314, ' is an island (no neighbors)')
('WARNING: ', 325, ' is an island (no neighbors)')
('WARNING: ', 372, ' is an island (no neighbors)')
('WARNING: ', 380, ' is an island (no neighbors)')
('WARNING: ', 398, ' is an island (no neighbors)')
('WARNING: ', 685, ' is an island (no neighbors)')
('WARNING: ', 952, ' is an island (no neighbors)')
('WARNING: ', 970, ' is an island (no neighbors)')
('WARNING: ', 1367, ' is an island (no neighbors)')
('WARNING: ', 1408, ' is an island (no neighbors)')
('WARNING: ', 1442, ' is an island (no neighbors)')
('WARNING: ', 1628, ' is an island (no neighbors)')
('WARNING: ', 1633, ' is an island (no neighbors)')
('WARNING: ', 1639, ' is an island (no neighbors)')
('WARNING: ', 1640, ' is an island (no neighbors)')
('WARNING: ', 2168, ' is an island (no neighbors)')


c:\Users\Jose\ids-vs-educacion-cdmx\venv\Lib\site-packages\esda\moran.py:1398: RuntimeWarning: invalid value encountered in divide
  self.z_sim = (self.Is - self.EI_sim) / self.seI_sim


Mapa econométrico LISA renderizado con éxito en 'docs/lisa_map.html'.
